In [3]:
import pandas as pd
import numpy as np
import requests
import time
from sklearn.linear_model import LinearRegression

In [4]:
cities = [
    {"city": "Denver", "latitude": 39.7392, "longitude": -104.9903},
    {"city": "Mumbai", "latitude": 19.0760, "longitude": 72.8777},
    {"city": "Reykjavik", "latitude": 64.1466, "longitude": -21.9426},
    {"city": "Tokyo", "latitude": 35.6762, "longitude": 139.6503},
    {"city": "Cairo", "latitude": 30.0444, "longitude": 31.2357},
    {"city": "São Paulo", "latitude": -23.5505, "longitude": -46.6333}
]

url = "https://archive-api.open-meteo.com/v1/archive"
start_date = "2016-01-01"
end_date = "2026-01-01"

In [5]:
import time
import requests
import pandas as pd

MAX_RETRIES = 5
INITIAL_BACKOFF = 5.0  # Wait 5s, 10s, 20s, 40s on retries
DELAY_BETWEEN_CITIES = 3.0  # Pause between city requests

all_dataframes = []

for item in cities:
    params = {
        "latitude": item["latitude"],
        "longitude": item["longitude"],
        "start_date": start_date,
        "end_date": end_date,
        "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
        "hourly": "relative_humidity_2m",
        "timezone": "auto"
    }

    data = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, params=params, timeout=60)


            if response.status_code == 429:     # Handle rate limiting explicitly (HTTP 429)
                # Use server-provided Retry-After header if available, else exponential backoff
                retry_after = response.headers.get("Retry-After")
                sleep_time = float(retry_after) if retry_after else INITIAL_BACKOFF * (2 ** (attempt - 1))
                print(f"[429 Rate Limit] {item['city']}: Waiting {sleep_time:.1f}s before retry (Attempt {attempt}/{MAX_RETRIES})...")
                time.sleep(sleep_time)
                continue
            response.raise_for_status()
            data = response.json()
            break  # Successful fetch
        except requests.exceptions.RequestException as e:
            sleep_time = INITIAL_BACKOFF * (2 ** (attempt - 1))
            print(f"[Attempt {attempt}/{MAX_RETRIES}] Error fetching {item['city']}: {e}")
            if attempt < MAX_RETRIES:
                print(f"Retrying in {sleep_time:.1f}s...")
                time.sleep(sleep_time)
            else:
                print(f"Skipping {item['city']} after {MAX_RETRIES} failed attempts.")

    # Parse valid response
    if data and "daily" in data and "hourly" in data:
        # Calculate daily relative humidity from hourly data
        hourly_times = pd.to_datetime(data["hourly"]["time"])
        humidity_df = pd.DataFrame({
            "date": hourly_times.date,
            "humidity": data["hourly"]["relative_humidity_2m"]
        })
        daily_humidity = humidity_df.groupby("date")["humidity"].mean().reset_index()
        daily_humidity["date"] = pd.to_datetime(daily_humidity["date"])

        # Base daily metrics
        city_df = pd.DataFrame({
            "date": pd.to_datetime(data["daily"]["time"]),
            "city": item["city"],
            "latitude": item["latitude"],
            "longitude": item["longitude"],
            "temp_min": data["daily"]["temperature_2m_min"],
            "temp_max": data["daily"]["temperature_2m_max"],
            "temp_mean": data["daily"]["temperature_2m_mean"],
            "precipitation_sum": data["daily"]["precipitation_sum"],
            "wind_speed": data["daily"]["wind_speed_10m_max"]
        })

        # Merge daily humidity into the city DataFrame on date
        city_df = pd.merge(city_df, daily_humidity, on="date", how="left")

        all_dataframes.append(city_df)
        print(f"Successfully ingested {item['city']}")

    # Pause briefly before querying the next city
    time.sleep(DELAY_BETWEEN_CITIES)

Successfully ingested Denver
Successfully ingested Mumbai
Successfully ingested Reykjavik
Successfully ingested Tokyo
[429 Rate Limit] Cairo: Waiting 5.0s before retry (Attempt 1/5)...
[429 Rate Limit] Cairo: Waiting 10.0s before retry (Attempt 2/5)...
Successfully ingested Cairo
Successfully ingested São Paulo


In [6]:
# 3. Combine into single DataFrame
if all_dataframes:
    df = pd.concat(all_dataframes, ignore_index=True).sort_values(["city", "date"]).reset_index(drop=True)

In [7]:
print(f"Combined Dataset shape: {df.shape}\n")
print(f"Cities included: {df['city'].unique().tolist()}\n")

Combined Dataset shape: (21924, 10)

Cities included: ['Cairo', 'Denver', 'Mumbai', 'Reykjavik', 'São Paulo', 'Tokyo']



In [8]:
df.isnull().sum()

,0
date,0
city,0
latitude,0
longitude,0
temp_min,0
temp_max,0
temp_mean,0
precipitation_sum,0
wind_speed,0
humidity,0


In [9]:
df.groupby("city")["date"].agg(["min","max","count"])

,min,max,count
city,,,
Cairo,2016-01-01,2026-01-01,3654
Denver,2016-01-01,2026-01-01,3654
Mumbai,2016-01-01,2026-01-01,3654
Reykjavik,2016-01-01,2026-01-01,3654
São Paulo,2016-01-01,2026-01-01,3654
Tokyo,2016-01-01,2026-01-01,3654


In [10]:
df.duplicated(subset=["city","date"]).sum()

np.int64(0)

In [11]:
def get_window_stats(df, city, doy, window=7):
    days = [(doy + i - 1) % 366 + 1 for i in range(-window, window+1)]
    subset = df[(df["city"] == city) & (df["day_of_year"].isin(days))]
    return subset["temp_mean"].mean(), subset["temp_mean"].std()

In [12]:
# 2. Build the historical baseline using the rolling window function
df["day_of_year"] = df["date"].dt.dayofyear
baseline_records = []
unique_cities = df["city"].unique()

print("Building rolling window baselines...")
for city in unique_cities:
    for doy in range(1, 367):  # Days 1 to 366
        mean_val, std_val = get_window_stats(df, city, doy, window=7)
        baseline_records.append({
            "city": city,
            "day_of_year": doy,
            "temp_mean_avg": mean_val,
            "temp_mean_std": std_val
        })

baseline_df = pd.DataFrame(baseline_records)

Building rolling window baselines...


In [13]:
# 3. Merge baseline stats back into main historical DataFrame
df = df.merge(baseline_df, on=["city", "day_of_year"], how="left")

In [14]:
# 4. Compute Z-Score and flag anomalies (|Z| > 2.0)
df["temp_zscore"] = (df["temp_mean"] - df["temp_mean_avg"]) / df["temp_mean_std"]
df["is_anomaly"] = df["temp_zscore"].abs() > 2.0

In [15]:
# 5. BACKTEST VALIDATION / SANITY CHECKS
total_rows = len(df)
total_anomalies = df["is_anomaly"].sum()
overall_pct = (total_anomalies / total_rows) * 100

print("\n================ BACKTEST RESULTS ================")
print(f"Total Rows Evaluated: {total_rows}")
print(f"Total Anomalies Flagged: {total_anomalies} ({overall_pct:.2f}% of total dataset)")

print("\nBreakdown per City:")
city_breakdown = df.groupby("city")["is_anomaly"].agg(
    total_days="count",
    anomalies="sum",
    anomaly_pct=lambda x: (x.sum() / x.count()) * 100
)
print(city_breakdown)


================ BACKTEST RESULTS ================
Total Rows Evaluated: 21924
Total Anomalies Flagged: 834 (3.80% of total dataset)

Breakdown per City:
           total_days  anomalies  anomaly_pct
city                                         
Cairo            3654        143     3.913519
Denver           3654        145     3.968254
Mumbai           3654        136     3.721949
Reykjavik        3654        118     3.229338
São Paulo        3654        151     4.132458
Tokyo            3654        141     3.858785
